# Mô hình dự đoán kết quả trận đấu trong giải Premier League

Dự đoán kết quả trận đấu dựa trên dữ liệu đã được làm sạch của các trận đấu trong giải EPL từ mùa giải 2017-2018 đến mùa 2024-2025.

In [3]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 29.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 34.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Mô hình dự đoán được sử dụng là Random Forest của thư viện scikit-learn. Dựa trên số bàn thắng qua các trận đấu, mô hình đưa ra dự đoán kết quả đội nhà "Thắng", "Hoà" hoặc "Thua" so với đội khách.

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load match data
matches = pd.read_csv('epl_matches_clean.csv')

# Ensure date is datetime and sort chronologically (crucial for form calculation)
matches['date'] = pd.to_datetime(matches['date'])
matches = matches.sort_values(['date', 'match_week']).reset_index(drop=True)

# Define match result target: 'H' (Home Win), 'A' (Away Win), 'D' (Draw)
def get_match_result(row):
    if row['home_goals'] > row['away_goals']:
        return 'H'
    elif row['home_goals'] < row['away_goals']:
        return 'A'
    else:
        return 'D'

matches['result'] = matches.apply(get_match_result, axis=1)
print(f"Total matches loaded: {len(matches)}")

Total matches loaded: 2970


## Feature Engineering

Sử dụng dữ liệu số bàn thắng của các trận đấu tạo thành 3 feature chính là phong độ 2 đội qua 5 trận đấu gần nhất và lịch sử đối đầu của 2 đội. Mỗi feature sẽ xét thêm yếu tố đội nhà đội khách.

#### Phong độ 5 trận gần nhất

Xét 5 trận đấu gần nhất để đánh giá phong độ của đội dựa trên số bàn ghi được và số bàn thủng lưới.

#### Lịch sử đối đầu

Xét lịch sử đối đầu của 2 đội (giới hạn 10 trận gần nhất) để đánh giá tương quan sức mạnh của 2 đội.

In [ ]:
def get_team_recent_form(df, team, current_date, n=5):
    # Get last n matches for the team prior to current_date
    past = df[(df['date'] < current_date) & ((df['home_team'] == team) | (df['away_team'] == team))]
    past = past.tail(n)
    
    if len(past) == 0:
        return 0.0, 0.0 # Default if no history
    
    goals_scored = []
    goals_conceded = []
    
    for _, row in past.iterrows():
        if row['home_team'] == team:
            goals_scored.append(row['home_goals'])
            goals_conceded.append(row['away_goals'])
        else:
            goals_scored.append(row['away_goals'])
            goals_conceded.append(row['home_goals'])
            
    return np.mean(goals_scored), np.mean(goals_conceded)

def get_head_to_head(df, home_team, away_team, current_date, n=10):
    # Past matches strictly between these two teams
    past = df[(df['date'] < current_date) & 
              (((df['home_team'] == home_team) & (df['away_team'] == away_team)) | 
               ((df['home_team'] == away_team) & (df['away_team'] == home_team)))]
    past = past.tail(n)
    
    if len(past) == 0:
        return 1.0, 1.0 # Neutral default
        
    home_goals_h2h = []
    away_goals_h2h = []
    
    for _, row in past.iterrows():
        if row['home_team'] == home_team:
            home_goals_h2h.append(row['home_goals'])
            away_goals_h2h.append(row['away_goals'])
        else:
            home_goals_h2h.append(row['away_goals'])
            away_goals_h2h.append(row['home_goals'])
            
    return np.mean(home_goals_h2h), np.mean(away_goals_h2h)

## Chuẩn bị dữ liệu để huấn luyện mô hình

In [16]:
features = []

for idx, row in matches.iterrows():
    date = row['date']
    home = row['home_team']
    away = row['away_team']
    season = row['season']
    
    # Recent 5 matches goal performance
    h_scored_5, h_conceded_5 = get_team_recent_form(matches, home, date, 5)
    a_scored_5, a_conceded_5 = get_team_recent_form(matches, away, date, 5)

    # Head-to-head past goals
    h_h2h_goals, a_h2h_goals = get_head_to_head(matches, home, away, date, 5)
    
    features.append({
        'game_id': row['game_id'],
        'home_scored_last5': h_scored_5,
        'home_conceded_last5': h_conceded_5,
        'away_scored_last5': a_scored_5,
        'away_conceded_last5': a_conceded_5,
        'h2h_home_goals': h_h2h_goals,
        'h2h_away_goals': a_h2h_goals,
        'result': row['result']
    })

feature_df = pd.DataFrame(features)

# Drop any rows with missing values so model_df is properly defined
model_df = feature_df.dropna().copy()

print(f"Feature dataset shape: {model_df.shape}")

Feature dataset shape: (2970, 8)


## Huấn luyện và đánh giá mô hình

Mô hình sử dụng dữ liệu từ các mùa giải (80%) trước để huấn luyện và đánh giá dựa trên các mùa giải gần đây (20%).

In [17]:
feature_cols = [
    'home_scored_last5', 'home_conceded_last5', 
    'away_scored_last5', 'away_conceded_last5', 
    'h2h_home_goals', 'h2h_away_goals'
]

X = model_df[feature_cols]
y = model_df['result']

# Chronological split (80% train, 20% test)
split_idx = int(len(model_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Train Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 47.98%

Classification Report:
              precision    recall  f1-score   support

           A       0.47      0.53      0.50       194
           D       0.22      0.06      0.09       139
           H       0.52      0.67      0.58       261

    accuracy                           0.48       594
   macro avg       0.40      0.42      0.39       594
weighted avg       0.43      0.48      0.44       594

